# 02 — Transform

**Purpose:** Clean, align, and merge raw datasets into analysis-ready tables saved to `data/processed/`.

## Steps
1. Load parquet files from `data/raw/`
2. Normalise date indices to a common monthly frequency
3. Compute derived series: yield curve spread (10y-2y), YoY inflation, real rates
4. Merge FRED, World Bank, and IMF into a unified panel
5. Forward-fill short gaps (≤2 months); flag longer gaps

## Outputs
- `data/processed/macro_panel.parquet` — unified monthly panel
- `data/processed/us_series.parquet` — US-only high-frequency series

## Papermill Parameters
- `run_date` — ISO date string injected by the GitHub Actions workflow

In [ ]:
run_date = None

In [ ]:
# Mount Google Drive for persistent storage (Colab only)
try:
    from google.colab import drive
    drive.mount("/content/drive")
    import os
    DRIVE_DIR = "/content/drive/MyDrive/macro-dashboard"
    if not os.path.exists("data"):
        os.symlink(f"{DRIVE_DIR}/data", "data")
    print("Google Drive mounted. Reading from:", DRIVE_DIR)
except Exception:
    print("Not in Colab — using local data/ directory.")

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "pandas", "pyarrow", "numpy"])
print("Packages ready.")

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW_DIR = Path("data/raw")
PROCESSED_DIR = Path("data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
print("Dirs ready.")

In [ ]:
# --- Load raw parquets ---
fred_raw = pd.read_parquet(RAW_DIR / "fred_series.parquet")
wb_raw   = pd.read_parquet(RAW_DIR / "worldbank.parquet")
imf_raw  = pd.read_parquet(RAW_DIR / "imf_weo.parquet")

print("FRED:  ", fred_raw.shape, "| index:", fred_raw.index.dtype)
print("WB:    ", wb_raw.shape,   "| index:", wb_raw.index.names)
print("IMF:   ", imf_raw.shape,  "| columns:", imf_raw.columns.tolist())

In [ ]:
# --- FRED: resample to month-end, derive indicators ---
fred = fred_raw.copy()
fred.index = pd.to_datetime(fred.index)
fred = fred.resample("ME").last()

# Yield curve spreads
fred["yield_spread_10y2y"] = fred["t10y"] - fred["t2y"]
fred["yield_spread_10y3m"] = fred["t10y"] - fred["t3m"]

# YoY inflation from index levels
fred["cpi_yoy_pct"] = fred["cpi_yoy"].pct_change(12) * 100
fred["pce_yoy_pct"] = fred["pce_yoy"].pct_change(12) * 100

# Approximate real 10y rate
fred["real_rate_10y"] = fred["t10y"] - fred["cpi_yoy_pct"]

# M2 YoY growth
fred["m2_yoy_pct"] = fred["m2"].pct_change(12) * 100

# Forward-fill short gaps only (≤2 months)
fred = fred.ffill(limit=2)

print("FRED monthly shape:", fred.shape)
print(fred[["yield_spread_10y2y", "cpi_yoy_pct", "real_rate_10y"]].tail(5))

In [ ]:
# --- World Bank: normalize to annual datetime index ---
wb = wb_raw.reset_index()
# date level may be string years or datetime — normalise to year-start datetime
wb["date"] = pd.to_datetime(wb["date"].astype(str).str[:4], format="%Y")
wb = wb.set_index(["country", "date"]).sort_index()
wb.columns = ["wb_" + c for c in wb.columns]
print("WB shape:", wb.shape)

# --- IMF WEO: pivot to wide format ---
imf = imf_raw.copy()
imf["date"] = pd.to_datetime(imf["TIME_PERIOD"].astype(str), format="%Y")
imf_wide = imf.pivot_table(
    index=["REF_AREA_LABEL", "date"],
    columns="CONCEPT_CODE",
    values="OBS_VALUE",
    aggfunc="first"
).rename_axis(index={"REF_AREA_LABEL": "country"})
imf_wide.columns = ["imf_" + c for c in imf_wide.columns]
print("IMF wide shape:", imf_wide.shape)

# --- Merge WB + IMF into global annual panel ---
global_panel = wb.join(imf_wide, how="outer").sort_index()
global_panel = global_panel.ffill(limit=1)
print("Global panel shape:", global_panel.shape)
print(global_panel.xs("United States", level="country").tail(3))

In [ ]:
# --- Save outputs ---
fred.to_parquet(PROCESSED_DIR / "us_series.parquet")
print(f"US series saved:    {fred.shape} → {PROCESSED_DIR}/us_series.parquet")

global_panel.to_parquet(PROCESSED_DIR / "macro_panel.parquet")
print(f"Global panel saved: {global_panel.shape} → {PROCESSED_DIR}/macro_panel.parquet")